# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

The problem we are addressing is to reliably **detecting and maintaining consistent identities for players and the ball** across multiple synchronized camera views.
Then, from the multi-view tracking results, we aim to perform **3D reconstruction** of player and ball positions, thanks to the multi-view geometry and camera calibration data.

Through different approaches, we aim to solve this problem, achieving accurate and robust tracking and reconstruction results.


---

### Imports

In [ ]:
from src.utils.video import open_video, get_frames
from src.utils.visualization import show_images, show_detection_table, show_identity_table, show_hota_table
from src.utils.model_loading import load_fine_tuned_yolo_model
from src.utils.annotations.download_annotations import download_annotations
from src.utils.annotations.load_annotations import load_annotations

from src.types.tracking import merge_detections

from src.detection.mog2_detection import run_mog2_detection, mog2_to_detection_output
from src.detection.yolo_detection import run_yolo_detection, yolo_to_detection_output
from src.tracking.sort import apply_sort
from src.tracking.deep_sort import apply_deep_sort
from src.detection.nms import class_independent_nms
from src.tracking.label_resolution import resolve_track_labels

from src.evaluation.evaluate_tracking import evaluate_tracking
from src.evaluation.evaluate_detection import evaluate_detection

from collections import defaultdict
import cv2
import time
import os
from pathlib import Path
from dotenv import load_dotenv
from ultralytics import YOLO

### Global Parameters

Global parameter variables configure experiment scope and file locations, enabling quick tests, dataset/model switching, and evaluation-scope changes without modifying implementation.

In [ ]:
INSPECT_CAMERA_ID = "cam_13"                                # The camera we will inspect in detail in the notebook
FRAME_STRIDE = 5                                            # Step between GT annotation frames and prediction frame indices
MAX_FRAMES   = None                                         # Frames to load per camera (None = all frames)

VIDEOS_DIR = "data/videos"                                  # Directory where the input videos are stored
CAMERA_SETTINGS_DIR = "data/camera_settings"                # Directory where the camera settings JSON files are stored
MODELS_DIR = "models/"                                      # Directory where pretrained and fine-tuned models are stored
FINE_TUNED_MODELS_DIR = f"{MODELS_DIR}/fine_tuned_models"   # Directory where fine-tuned models will be saved
ANNOTATIONS = "data/annotations/tracking_01"                # Directory where the tracking annotations are stored (after downloading and extraction)

# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json",
    },
}

### Open Video and Read Frames

Load the synchronized multi-camera video recordings of a basketball game captured at the **Sanbapolis facility** in Trento, Italy. These are the source videos used for all subsequent processing and evaluation.

All videos capture a single game act and were recorded at 25 frames per second (25 fps).



In [ ]:
frames_color = {}
fps = {}
for cam_id, cam in CAMERAS.items():
    # Load the video 
    cap = open_video(cam["video_path"])

    # Extract the frames
    frames_color[cam_id], _ = get_frames(cap, max_frames=MAX_FRAMES)  

    # Get the frames per second (fps) of the video
    fps[cam_id] = cap.get(cv2.CAP_PROP_FPS)

    # Release the video capture object to free resources
    cap.release()
    print(f"[{cam_id}] Loaded {len(frames_color[cam_id])} frames at {fps[cam_id]:.1f} fps")

show_images(frames_color[INSPECT_CAMERA_ID])

---

# Detection


---

## Background Subtraction with MOG2 for Player Detection

> Manafifard, M., Ebadi, H., & Abrishami Moghaddam, H. (2017). A survey on player tracking in soccer videos. *Computer Vision and Image Understanding*

Following the survey by Manafifard et al. (2017), we adopt background subtraction as our first detection approach, given its established effectiveness as widely-used baseline before exploring learned-based methods.

Specifically, we use the MOG2 (Mixture of Gaussians v2) background subtractor to detect moving players in each frame.
MOG2 models each pixel's intensity distribution as a mixture of $K$ Gaussian components:

$$P(x_t) = \sum_{k=1}^{K} w_{k,t} \cdot \mathcal{N}(x_t \mid \mu_{k,t}, \sigma_{k,t}^2)$$

A pixel is classified as **foreground** if the Mahalanobis distance to the best-matching background Gaussian exceeds the threshold (VAR_THRESHOLD), otherwise it is **background**.

MOG2 improves over the original MOG in three key ways:
1. **Adaptive parameters**: Weights, means, and variances are updated per-frame using an adaptive learning rate, allowing fast adaptation to slow lighting changes without overreacting to sudden foreground.
2. **Shadow detection**: MOG2 explicitly detects shadowsand marks them as background rather than foreground, reducing false positives from cast shadows on the field.
3. **Better stability**: Multiple Gaussians per pixel better capture multimodal backgrounds (e.g., in our case the spectators), reducing the drift and variance instability present in the original MOG.

Also other methods such as KNN, LSBP and GSOC were tried, but MOG2 was found to be equally effective while being more computationally efficient and simpler, making it a suitable choice for our baseline detection approach.

We further **improved the detection** by applying pre-processing steps and post-processing steps to the frames and the foreground masks, respectively.


For **pre-processing**, we applied a illumination normalization technique called **CLAHE (Contrast Limited Adaptive Histogram Equalization)** to enhance the contrast of the frames, which helps in better foreground detection.

For **post-processing**, we applied morphological operations such as opening and closing to remove noise and fill holes in the detected foreground masks, and then we applied contour detection to extract the bounding boxes of the detected players. We removed small contours that are likely to be noise by filtering based on contour area.

### Parameters

The parameters below were tuned specifically for **cam_13** through empirical experiments. This reveals a fundamental limitation of background subtraction: **the method lacks cross-camera generalization**.

In [ ]:
MOG2_LEARNING_RATE  = -1        # -1 lets OpenCV adapt the learning rate automatically
MOG2_HISTORY        = 1000      # number of frames used to model the background
MOG2_VAR_THRESHOLD  = 25        # Mahalanobis distance threshold for foreground/background decision
MOG2_OPENING_KERNEL = 7         # morphological opening kernel size (noise removal)
MOG2_CLOSING_KERNEL = 11        # morphological closing kernel size (hole filling)
MOG2_MIN_AREA       = 1500      # minimum contour area to keep (px²)
MOG2_MAX_AREA       = 200_000   # maximum contour area to keep (px²)

### Run MOG2 Detection on All Frames

For each camera, the code runs MOG2 background subtraction on all frames (using the supplied parameters), producing cleaned foreground masks and measuring elapsed time.
It then converts those masks to a `DetectionOutput` (per-frame bounding boxes + metadata, using the camera’s fps)

In [ ]:
mog2_masks = {}
mog2_detection_output = {}

for cam_id in CAMERAS:
    
    # Run MOG2 detection 
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with MOG2 detection...")
    start_time = time.time()
    mog2_masks[cam_id] = run_mog2_detection(
        frames_color[cam_id],
        learning_rate=MOG2_LEARNING_RATE,
        history_length=MOG2_HISTORY,
        var_threshold=MOG2_VAR_THRESHOLD,
        opening_kernel_size=MOG2_OPENING_KERNEL,
        closing_kernel_size=MOG2_CLOSING_KERNEL,
        min_area=MOG2_MIN_AREA,
        max_area=MOG2_MAX_AREA,
    )
    elapsed = time.time() - start_time

    # Convert MOG2 masks to DetectionOutput format for evaluation and tracking
    mog2_detection_output[cam_id] = mog2_to_detection_output(mog2_masks[cam_id], camera_id=cam_id, fps=fps[cam_id])
    n_dets = sum(f.num_detections for f in mog2_detection_output[cam_id].frames)
    print(f"  Done in {elapsed:.1f}s — {n_dets} detections across {len(mog2_detection_output[cam_id].frames)} frames")

### Inspect MOG2 Detections

We visualize the MOG2 blob detected by outputting the binary mask and then the bounding boxes of the detected players. 

In [ ]:
# Show raw masks for the inspection camera (skip warm-up frames)
show_images(mog2_masks[INSPECT_CAMERA_ID][15:])

Despite MOG2's improvements, several problems are apparent in the detected blobs above:
- **Undetected reflective shadows**: Even with built-in shadow detection, shadows persist on the field because the floor's high reflectivity produces shadows with non-classical hue values. 
- **Shape loss from aggresive morphology**: The morphological operations used to clean the masks can sometimes be too aggressive, leading to loss of parts of the player blobs, especially for players in motion or partially occluded. 
- **Static background assumption**: MOG2 assumes a mostly static background, which is violated in our case due to the presence of moving spectators and dynamic lighting conditions, leading to false positives and missed detections.
- **Static player**: MOG2 relies on motion to detect foreground objects, so players that are static or moving very slowly may not be detected, leading to missed detections during moments of inactivity or slow movement. 
An example could be the second referee.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], mog2_detection_output[INSPECT_CAMERA_ID])

As expected, these limitations collectively generate **inconsistent and unreliable bounding boxes**, which would lead to poor tracking performance in subsequent steps.

---

## Pre-Trained YOLOv11

> Jocher, G., Qiu, J., & Chaurasia, A. (2023). Ultralytics YOLO. 
>
> Joseph Redmon and Santosh Divvala and Ross Girshick and Ali Farhadi. (2016). You Only Look Once: Unified, Real-Time Object Detection.

In order to overcome the limitations of background subtraction, we adopt a deep learning-based object detection approach using the **YOLOv11 model**.

YOLO (You Only Look Once) is a family of single-stage, real-time detectors that predict bounding boxes and class probabilities directly from image pixels using a **convolutional backbone** and a detection head. Modern YOLO variants balance accuracy and speed, making them suitable for per-frame video inference.

We choosed a learning-based model because those approaches generalize better across different cameras, viewpoints, and lighting conditions: they learn semantic features (shape, texture, context) that are less sensitive to camera-specific artifacts than background-subtraction heuristics.

We started from a **pre-trained YOLOv11 model** in order to leverage the general object detection capabilities it has learned from large-scale datasets, and avoiding to train a model from scratch given the limited size of our dataset. 


### Parameters

The parameters for the YOLOv11 model are mostly related to inference settings, such as confidence threshold and non-maximum suppression (NMS) threshold, which control the trade-off between precision and recall in the detections.

In [ ]:
YOLO_PRETRAINED_CONF = 0.4    # confidence threshold
YOLO_PRETRAINED_IOU  = 0.45   # IoU threshold for built-in NMS
YOLO_PRETRAINED_SIZE = 640    # inference image size

### Models

In this section we load the `yolov11m.pt` model, which is a medium-sized variant of YOLOv11 that offers a good balance between accuracy and inference speed. This model was pre-trained on the COCO dataset, which includes a "person" class that is relevant for our player detection task.

In [ ]:
# Resolve the path of pretrained yolo model
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)
pretrained_yolo_model_path = os.path.join(MODELS_DIR, "yolo11m.pt")

# Load the YOLOv11m model
pretrained_yolo_model = YOLO(pretrained_yolo_model_path)    

### Run YOLO Detection on All Frames

As before, we run inference on all frames for each camera, measuring elapsed time and counting detections and outputting the results in a `DetectionOutput` format.

We use YOLOv11's built-in `.predict()` method to detect objects in each frame. The output will include bounding boxes, confidence scores, and class labels for each detected object.


In [ ]:
yolo_pretrained_raw = {}

for cam_id in CAMERAS:
    # Run YOLOv11m detection
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with pre-trained YOLO detection...")
    start_time = time.time()
    yolo_pretrained_raw[cam_id] = run_yolo_detection(
        pretrained_yolo_model, frames_color[cam_id],
        conf_threshold=YOLO_PRETRAINED_CONF,
        iou_threshold=YOLO_PRETRAINED_IOU,
        inference_size=YOLO_PRETRAINED_SIZE,
    )
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s — {sum(len(r.boxes) for r in yolo_pretrained_raw[cam_id])} raw detections")

In [ ]:
yolo_pretrained_detection_output = {}
for cam_id in CAMERAS:
    # Convert YOLO raw detections to DetectionOutput format for evaluation and tracking
    yolo_pretrained_detection_output[cam_id] = yolo_to_detection_output(
        yolo_pretrained_raw[cam_id], pretrained_yolo_model,
        camera_id=cam_id, fps=fps[cam_id], source="yolo_v11m_pt",
    )

    # Count total detections after conversion
    n_dets = sum(f.num_detections for f in yolo_pretrained_detection_output[cam_id].frames)
    print(f"[{cam_id}] Total detections after conversion: {n_dets} across {len(yolo_pretrained_detection_output[cam_id].frames)} frames")

### Inspect YOLO Detections

We visualize the YOLO detections by drawing the predicted bounding boxes on the frames and displaying the confidence scores and class labels.

In [ ]:
cam = INSPECT_CAMERA_ID
print(f"First frame detections [{cam}]: {yolo_pretrained_detection_output[cam].frames[0].num_detections} objects detected")
for i, det in enumerate(yolo_pretrained_detection_output[cam].frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={det.class_name}, Confidence={det.confidence:.2f}, BBox={det.bbox}")

show_images(frames_color[cam], yolo_pretrained_detection_output[cam])

As expected, the YOLO detections are much more accurate and consistent than the MOG2 blobs, with fewer false positives and better localization of the players. 

However, we can still observe that is unable to detect the ball, which is a small and fast-moving object that can be easily missed by the model, especially if it is partially occluded or blurred. This highlights the need for further improvements in the detection stage.

Then, the player and referee are detected as `person`, making it impossible to distinguish between them based on the class label alone. This is a common limitation of general object detectors when applied to specific domains.

---

## Fine-Tuned YOLOv11

> Jocher, G., Qiu, J., & Chaurasia, A. (2023). Ultralytics YOLO. [`yolo`]
>
> J. Redmon and S. Divvala and R. Girshick and Ali Farhadi. (2016). You Only Look Once: Unified, Real-Time Object Detection.

The pre-trained model delivers reliable bounding box detection for people, but carries fundamental limitations in our domain. To overcome both gaps we **fine-tune** the YOLOv11m backbone on other annotated dataset from the same Sanbapolis facility. 

Fine-tuning retains the general visual representations learned at scale (edges, textures, body silhouettes) and replaces the classification head with one trained on our domain-specific class set: one class per player identity (e.g. `Red_11`, `White_7`) and a dedicated `Ball` class. 

However, to address the ball detection gap, we also adopt a **multi-pass inference strategy** that runs two separate passes over each frame at different resolutions and class restrictions.                                             

### Parameters

In [ ]:
YOLO_FINETUNED_PLAYER_CONF = 0.3    # confidence threshold for player pass
YOLO_FINETUNED_BALL_CONF   = 0.1    # lower threshold for ball (small object, favour recall)
YOLO_FINETUNED_PLAYER_SIZE = 640    # inference size for player pass
YOLO_FINETUNED_BALL_SIZE   = 1280   # higher resolution for ball pass

### Models

Using a script, we load from **HuggingFace Hub** the fine-tuned model weights, uploaded by us after training on the annotated Sanbapolis dataset. 

In [ ]:
# Script to load the fine-tuned YOLO model for player pass detection
yolo_finetuned_model = load_fine_tuned_yolo_model()

### Run Fine-Tuned YOLO Detection on All Frames

As before, we run inference on all frames for each camera, measuring elapsed time and counting detections and outputting the results in a `DetectionOutput` format.

In [ ]:
yolo_finetuned_player_raw = {}
for cam_id in CAMERAS:
    # Run fine-tuned YOLO detection for player pass
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with fine-tuned YOLO (player pass)...")
    start_time = time.time()
    yolo_finetuned_player_raw[cam_id] = run_yolo_detection(
        yolo_finetuned_model,
        frames_color[cam_id],
        conf_threshold=YOLO_FINETUNED_PLAYER_CONF,
        inference_size=YOLO_FINETUNED_PLAYER_SIZE,
        class_ids=list(range(1, len(yolo_finetuned_model.names))),
    )
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s — {sum(len(r.boxes) for r in yolo_finetuned_player_raw[cam_id])} player detections")

### Higher-Resolution Pass - Ball Only

At `imgsz = 640`, YOLOv11's backbone downsamples a 1080p frame by roughly 1.7×, so a 15 px basketball occupies fewer than two grid cells in the final feature map, far below the **minimum spatial footprint a detection head can reliably localize**.

**Doubling the inference resolution** to 1280 halves the effective downsampling ratio, increasing the grid-cell density dedicated to small objects and giving the head enough spatial signal to produce a confident, well-centred box.

We pair the higher resolution with a lower confidence threshold (0.1 vs. 0.3 for players). 

The ball is frequently motion-blurred, partially occluded, or caught mid-flight at an angle that reduces its visual similarity to the training patches.
Relaxing the threshold recovers these marginal but valid detections at the cost of a modest increase in false positives.


In [ ]:
yolo_finetuned_ball_raw = {}
for cam_id in CAMERAS:
    # Run fine-tuned YOLO detection for ball pass
    print(f"[{cam_id}] Processing {len(frames_color[cam_id])} frames with fine-tuned YOLO (ball pass)...")
    start_time = time.time()
    yolo_finetuned_ball_raw[cam_id] = run_yolo_detection(
        yolo_finetuned_model, frames_color[cam_id],
        conf_threshold=YOLO_FINETUNED_BALL_CONF,
        inference_size=YOLO_FINETUNED_BALL_SIZE,
        class_ids=[0],
    )
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s — {sum(len(r.boxes) for r in yolo_finetuned_ball_raw[cam_id])} ball detections")

### Merge Player and Ball Detections

Combine both passes into a single `DetectionOutput`, frame by frame. After fine-tuning, only the `class_ids` lists in the two passes above need to change, the merge stays identical.

In [ ]:
yolo_finetuned_merged_output = {}
for cam_id in CAMERAS:
    # Convert player and ball output
    player_out = yolo_to_detection_output(
        yolo_finetuned_player_raw[cam_id], yolo_finetuned_model,
        camera_id=cam_id, fps=fps[cam_id], source="yolo_v11m_ft",
    )
    ball_out = yolo_to_detection_output(
        yolo_finetuned_ball_raw[cam_id], yolo_finetuned_model,
        camera_id=cam_id, fps=fps[cam_id], source="yolo_v11m_ft",
    )

    # Merge both converted outputs
    yolo_finetuned_merged_output[cam_id] = merge_detections(player_out, ball_out)

    # Count total detections after merging
    n_merged = sum(f.num_detections for f in yolo_finetuned_merged_output[cam_id].frames)
    print(f"[{cam_id}] Merged: {n_merged} detections across {len(yolo_finetuned_merged_output[cam_id].frames)} frames")

### Inspect Merged Detections

We visualize the merged detections by drawing the predicted bounding boxes for both players and the ball on the frames, displaying the confidence scores and class labels.

In [ ]:
cam = INSPECT_CAMERA_ID
print(f"First frame detections [{cam}]: {yolo_finetuned_merged_output[cam].frames[0].num_detections} objects detected")
for i, det in enumerate(yolo_finetuned_merged_output[cam].frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={det.class_name}, Confidence={det.confidence:.2f}, BBox={det.bbox}")

show_images(frames_color[cam], yolo_finetuned_merged_output[cam])

As expected, the fine-tuned model delivers much more accurate and consistent detections for both players and the ball, with significantly fewer false positives and better localization.

However, YOLOs **built-in NMS operates independently within each class**.
Because our fine-tuned model encodes player identity directly in the class label (e.g. `Red_5`, `Red_11`), two overlapping boxes predicted for the same physical player but under different class names are treated as entirely distinct objects and both survive suppression. These duplicate boxes would then be handed to the downstream tracker as separate detections, spawning parallel tracks for the same person and fragmenting what should be a single stable identity across multiple track IDs.


---

## Class-Agnostic NMS

> Neubeck, A., & Van Gool, L. (2006). Efficient non-maximum suppression. *ICPR'06*, Vol. 3, 850–855. [`nms`]

Non-Maximum Suppression (NMS) is a standard post-processing step applied after object detection to **eliminate redundant overlapping boxes** for the same physical object.

The algorithm sorts all candidate boxes by confidence score, keeps the highest-scoring box, and suppresses any remaining box whose IoU with it exceeds a threshold $\tau$, repeating until no candidates remain.

To fix the previous issue, we therefore apply a **class-agnostic variant**: the same greedy suppression algorithm, but treating all detections as a single pool regardless of class. 


### Parameters

In [ ]:
NMS_IOU_THRESHOLD = 0.5    # IoU above which a lower-confidence box is suppressed

### Run YOLO Detection with Class-Agnostic NMS on All Frames

We take the merged detections from the previous step and apply a class-agnostic NMS to eliminate duplicate boxes for the same physical player, regardless of their predicted class labels.

In [ ]:
yolo_finetuned_detection_output = {}
for cam_id in CAMERAS:
    # Apply class-agnostic NMS to the merged detections
    before = sum(len(f.detections) for f in yolo_finetuned_merged_output[cam_id].frames)
    yolo_finetuned_detection_output[cam_id] = class_independent_nms(yolo_finetuned_merged_output[cam_id], iou_threshold=NMS_IOU_THRESHOLD)
    after = sum(len(f.detections) for f in yolo_finetuned_detection_output[cam_id].frames)
    print(f"[{cam_id}] Class-agnostic NMS: {before} → {after} detections ({before - after} suppressed)")

### Inspect YOLO Detections with Class-Agnostic NMS

We visualize the final detections after class-agnostic NMS by drawing the predicted bounding boxes on the frames and displaying the confidence scores and class labels.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], yolo_finetuned_detection_output[INSPECT_CAMERA_ID])

Class-agnostic NMS successfully collapses duplicate boxes for the same physical player, resulting in cleaner detections that are more suitable for tracking. The ball detections remain unaffected since they are distinct objects with their own class label.

We consider this the **final output** of the detection stage, which will be fed into the tracking stage in the next part of the notebook.

---

# Tracking

---

## SORT Tracking

> Bewley, A., Ge, Z., Ott, L., Ramos, F., & Upcroft, B. (2016). Simple online and realtime tracking. *ICIP 2016*, 3464–3468. [`sort`]

SORT (Simple Online and Realtime Tracking) assigns a stable track_id to each detection through a three-step loop executed every frame: **predict**, **match**, **update**.
                                 
Each active track maintains a **Kalman filter** whose state vector encodes bounding-box position and velocity $\mathbf{x} = [u, v, s, r, \dot{u},\dot{v}, \dot{s}]^\top$, where $(u, v)$ is the box centre, $s$ its scale (area), and $r$ the fixed aspect ratio. At the start of each frame the filter propagates every track forward under a constant-velocity motion model, producing a predicted box location even before any detection is observed.

Predicted boxes are then matched to incoming detections using the **Hungarian algorithm** on a cost matrix built from pairwise IoU: detections and predictions with IoU below a minimum threshold are never matched (gating), preventing long-range associations caused by Kalman drift.
Matched tracks absorb their detection as a measurement update, unmatched detections spawn new tentative tracks, and tracks that go unmatched for more than `max_age` frames are killed.


### Parameters

In [ ]:
SORT_MAX_IOU_DISTANCE = 0.8   # IoU gating threshold for Kalman-prediction-to-detection assignment
SORT_MAX_AGE          = 60    # frames a track stays alive while unmatched (~2.4 s at 25 fps)
SORT_N_INIT           = 2     # consecutive matches required before a track is confirmed

### Run SORT Tracking on All Frames

We take the final detections from the previous step and feed them into the SORT tracker, performing the **predict-match-update** loop for each frame and outputting a `TrackingOutput` containing the track IDs, bounding boxes, confidence scores, and class labels for each tracked object in each frame.

In [ ]:
sort_tracking_output = {}
for cam_id in CAMERAS:
    # Apply SORT tracking to the fine-tuned YOLO detections
    print(f"[{cam_id}] Running SORT on {len(yolo_finetuned_detection_output[cam_id].frames)} frames...")
    start_time = time.time()
    sort_tracking_output[cam_id] = apply_sort(
        yolo_finetuned_detection_output[cam_id],
        max_iou_distance=SORT_MAX_IOU_DISTANCE,
        max_age=SORT_MAX_AGE,
        n_init=SORT_N_INIT,
    )
    elapsed = time.time() - start_time

    # Count total tracks and detections after tracking
    total_tracks = len({d.track_id for f in sort_tracking_output[cam_id].frames for d in f.detections})
    total_dets   = sum(len(f.detections) for f in sort_tracking_output[cam_id].frames)
    print(f"  Done in {elapsed:.1f}s — {total_tracks} tracks across {total_dets} tracked detections")

### Inspect SORT Tracks

We visualize the final tracks by drawing the predicted bounding boxes for both players and the ball on the frames, displaying the confidence scores, class labels, and track IDs.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], sort_tracking_output[INSPECT_CAMERA_ID])

### Resolve Per-Track Labels

YOLO's class names already encode identity (e.g. `Red_11` = Red team #11), but the model occasionally flips a player's label between frames or returns two boxes with the same class in one frame. We use the track ids assigned above to vote for a stable label per track and resolve collisions where two tracks claim the same identity.

The track with the highest cumulative confidence keeps its top-choice label; any other track that wanted that label falls through to its next-best class. `Ball` is exempt, since it's a category, not a unique identity.

In [ ]:
sort_resolved_output = {}
for cam_id in CAMERAS:
    # Resolve track labels by majority voting on the detections associated with each track
    sort_resolved_output[cam_id] = resolve_track_labels(sort_tracking_output[cam_id])

    # Analyze label stability per track (expecting most tracks to have a single stable label, ideally "Ball" or "Player")
    per_track_label = defaultdict(set)
    for fd in sort_resolved_output[cam_id].frames:
        for d in fd.detections:
            per_track_label[d.track_id].add(d.class_name)
    
    # Identify tracks that still have multiple labels (should ideally be only the ball track if the model is confused between ball and player)
    multi_label = [tid for tid, labels in per_track_label.items() if len(labels) > 1]

    # Print summary of label stability
    print(f"[{cam_id}] Tracks with a single stable label: {len(per_track_label) - len(multi_label)} / {len(per_track_label)}")
    if multi_label:
        print(f"  Tracks still showing multiple labels (should only be Ball): {multi_label}")

### Inspect Resolved Tracks

We visualize the final tracks, with resolved labels, by drawing the predicted bounding boxes for both players and the ball on the frames, displaying the confidence scores, class labels, and track IDs.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], sort_resolved_output[INSPECT_CAMERA_ID])

---

## Tracking with DeepSORT

> Wojke, N., Bewley, A., & Paulus, D. (2017). Simple online and realtime tracking with a deep association metric. *arXiv:1703.07402*. [`deepsort`]
>
> Zhou, K., Yang, Y., Cavallaro, A., & Xiang, T. (2019). Omni-Scale Feature Learning for Person Re-Identification. *arXiv:1905.00953*. [`reid`]

DeepSORT retains the Kalman filter and Hungarian assignment from SORT and **extends** them with a deep appearance metric. 

For each detection, a lightweight **ReID network** (OSNet) extracts a compact embedding from the cropped box region.
Each track maintains a gallery of up to feature_budget past embeddings; the cosine distance between a new detection's embedding and a track's gallery mean is used as a second cost signal alongside IoU.

Matching is performed in a **cascade**: tracks are prioritized by recency (most recently seen first), and appearance distance is the primary gate, IoU is only used as a fallback for tracks that have just been created or that the appearance model is uncertain about. This ordering prevents a long-occluded track from hijacking a detection via a lucky IoU overlap, which is the primary failure mode of SORT during player crossings.

The **practical consequence** in sports footage is significantly fewer identity  switches: when two players cross and one is temporarily occluded, the appearance embedding of the re-emerging player pulls the correct track back even if the predicted Kalman box has drifted onto the wrong person.


### Parameters
The parameters below were tuned for sports footage at 25 fps:

- `max_age`: frames a track stays alive while unmatched before being killed. 30 
≈ 1.2 s of occlusion tolerance, too short for basketball where players cross
for 2–3 s. We use 60 (≈ 2.4 s).                                               
- `max_iou_distance`: IoU gating threshold. A looser value (0.8) tolerates more
Kalman drift over longer gaps; too loose causes id swaps between nearby       
players.
- `max_appearance_distance`: cosine distance gate for the OSNet appearance      
embedding. Lower values enforce stricter appearance matching.                 
- `n_init`: consecutive frames a tentative track must match before being
confirmed as active.      

In [ ]:
DEEPSORT_MAX_IOU_DISTANCE        = 0.8   # IoU gating. Looser (0.8) tolerates more Kalman drift over
DEEPSORT_MAX_APPEARANCE_DISTANCE = 0.2   # cosine distance threshold for appearance-based matching (lower = stricter, 0.2 is a common default)
DEEPSORT_MAX_AGE                 = 60    # frames a track stays alive while unmatched before being killed. 30 ≈ 1.2 s of occlusion tolerance — too short for football where players cross for 2–3 s. We use 60 (≈ 2.4 s).
DEEPSORT_N_INIT                  = 2     # consecutive matches required before a track is confirmed
DEEPSORT_FEATURE_BUDGET          = 100   # max appearance features stored per track

### Run DeepSORT Tracking on All Frames

We take the final detections from the previous step and feed them into our DeepSORT implementation as before.

In [ ]:
deepsort_tracking_output = {}
for cam_id in CAMERAS:
    # Apply DeepSORT tracking to the fine-tuned YOLO detections
    print(f"[{cam_id}] Running DeepSORT on {len(yolo_finetuned_detection_output[cam_id].frames)} frames...")
    start_time = time.time()
    deepsort_tracking_output[cam_id] = apply_deep_sort(
        yolo_finetuned_detection_output[cam_id],
        frames=frames_color[cam_id],
        max_iou_distance=DEEPSORT_MAX_IOU_DISTANCE,
        max_appearance_distance=DEEPSORT_MAX_APPEARANCE_DISTANCE,
        max_age=DEEPSORT_MAX_AGE,
        n_init=DEEPSORT_N_INIT,
        feature_budget=DEEPSORT_FEATURE_BUDGET,
    )
    elapsed = time.time() - start_time

    # Count total tracks and detections after tracking
    total_tracks = len({d.track_id for f in deepsort_tracking_output[cam_id].frames for d in f.detections})
    total_dets   = sum(len(f.detections) for f in deepsort_tracking_output[cam_id].frames)
    print(f"  Done in {elapsed:.1f}s — {total_tracks} tracks across {total_dets} tracked detections")

### Inspect DeepSORT Tracks

We visualize the final tracks by drawing the predicted bounding boxes for both players and the ball on the frames, displaying the confidence scores, class labels, and track IDs.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], deepsort_tracking_output[INSPECT_CAMERA_ID])

### Resolve Per-Track Labels

As with SORT, we use the track ids assigned above to vote for a stable label per track and resolve collisions where two tracks claim the same identity.

In [ ]:
deepsort_resolved_output = {}
for cam_id in CAMERAS:
    # Resolve track labels by majority voting on the detections associated with each track
    deepsort_resolved_output[cam_id] = resolve_track_labels(deepsort_tracking_output[cam_id])
    
    # Analyze label stability per track (expecting most tracks to have a single stable label, ideally "Ball" or "Player")
    per_track_label = defaultdict(set)
    for fd in deepsort_resolved_output[cam_id].frames:
        for d in fd.detections:
            per_track_label[d.track_id].add(d.class_name)

    # Identify tracks that still have multiple labels (should ideally be only the ball track if the model is confused between ball and player)
    multi_label = [tid for tid, labels in per_track_label.items() if len(labels) > 1]
    print(f"[{cam_id}] Tracks with a single stable label: {len(per_track_label) - len(multi_label)} / {len(per_track_label)}")
    if multi_label:
        print(f"  Tracks still showing multiple labels (should only be Ball): {multi_label}")

### Inspect Resolved Labels

We visualize the final tracks, with resolved labels, by drawing the predicted bounding boxes for both players and the ball on the frames, displaying the confidence scores, class labels, and track IDs.

In [ ]:
show_images(frames_color[INSPECT_CAMERA_ID], deepsort_resolved_output[INSPECT_CAMERA_ID])

---

# Evaluation

---

## Ground Truth

The code downloads annotations from Roboflow (if not already cached locally) using an API key stored in `.env`, extracts them into `ANNOTATIONS`, and loads one `DetectionOutput` per camera that the evaluators will use as the reference signal.

Ground truth annotations are **sparse**: the dataset provides bounding boxes every `FRAME_STRIDE` frames (every 5 frames at 25 fps). Both the detection and tracking evaluators receive `frame_stride` as a parameter and use it to align predicted frames with annotated frames, skipping non-annotated frames entirely. This ensures metrics are computed only where ground truth exists, avoiding artificial false positives on unannotated frames.

In [ ]:
# Download annotations if not already present (requires ANNOTATIONS_API_KEY in .env)
if not any(Path(ANNOTATIONS).glob("*.json")):
    load_dotenv()
    api_key = os.environ["ANNOTATIONS_API_KEY"]
    download_annotations("tracking-rybv7", "tracking_01", 1, api_key, ANNOTATIONS)
else:
    print(f"Annotations already present in {ANNOTATIONS} — skipping download.")

# Load annotations for each camera and print summary
ground_truth = {}
for cam_id in CAMERAS:
    ground_truth[cam_id] = load_annotations(camera_id=cam_id, version="tracking_01")
    n_ann = sum(len(f.detections) for f in ground_truth[cam_id].frames)
    print(f"[{cam_id}] Loaded {n_ann} annotations across {len(ground_truth[cam_id].frames)} frames")

---

## Detection

Detection metrics are computed at the bounding-box level, ignoring identity. A predicted box is a **True Positive (TP)** when its IoU with an unmatched ground-truth box meets or exceeds 0.5; otherwise it is a **False Positive (FP)**. A ground-truth box with no matching prediction is a **False Negative (FN)**.

**IoU (Intersection over Union)** is the area of overlap between the predicted box and the ground-truth box divided by the area of their union. A **common threshold** for considering a detection correct is IoU ≥ 0.5.

$$\text{IoU}(A, B) = \frac{A \cap B}{A \cup B}$$

Where $A$ is the predicted bounding box and $B$ is the ground-truth bounding box.

**Precision** measures what fraction of all predictions are actually correct:

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall** measures what fraction of all ground-truth objects were found:

$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1** is the harmonic mean of precision and recall, penalising any imbalance between the two — a detector that achieves one at the expense of the other will score poorly:

$$F_1 = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

**Mean IoU** is the average spatial overlap of matched TP pairs. It is independent of precision and recall and quantifies how tightly predicted boxes are localised around the ground truth.

In [ ]:
# Evaluate detection performance of MOG2, pre-trained YOLO, and fine-tuned YOLO against ground truth annotations
detection_results = {}
for method_name, outputs in [
    ("MOG2",             mog2_detection_output),
    ("YOLO pre-trained", yolo_pretrained_detection_output),
    ("YOLO fine-tuned",  yolo_finetuned_detection_output),
]:
    for cam_id in CAMERAS:
        detection_results[(method_name, cam_id)] = evaluate_detection(
            ground_truth[cam_id], outputs[cam_id], frame_stride=FRAME_STRIDE,
        )

# Display detection results in a table
show_detection_table(detection_results)

---

## Tracking

> Luiten, J., et al. (2020). HOTA: A Higher Order Metric for Evaluating Multi-object Tracking. *IJCV*, 129(2), 548–578. [`hota`]

Tracking evaluation uses two complementary metric families on top of the detection scores above.

### IDF1

IDF1 measures how consistently each ground-truth identity is covered by a single predicted track over time. For each ground-truth identity, the predicted track that covers the most of its frames is designated the best match.

- **IDTP**: frames where the best-matching predicted track is present and covers the correct identity.
- **IDFP**: frames where the predicted track is present but assigned to the wrong identity.
- **IDFN**: frames where the ground-truth identity is not covered by any consistent track.

$$\text{IDF1} = \frac{2 \cdot \text{IDTP}}{2 \cdot \text{IDTP} + \text{IDFP} + \text{IDFN}}$$

A low IDF1 signals frequent identity switches, the tracker reassigns a player to a new track ID rather than maintaining the original one across time.

### HOTA

HOTA decomposes overall tracking quality into independent detection and association components, evaluated at multiple IoU thresholds $\alpha \in \{0.05, 0.10, \dots, 0.95\}$.

**Detection accuracy** at threshold $\alpha$ is the IoU over all detections it rewards finding objects and penalises both missed and spurious boxes:

$$\text{DetA}(\alpha) = \frac{TP(\alpha)}{TP(\alpha) + FP(\alpha) + FN(\alpha)}$$

**Association accuracy** measures how well matched detections are linked into consistent tracks. For each matched ground-truth–prediction pair $(c, \hat{c})$, it computes a per-pair IoU over all frames where that pair is active and averages across all matched pairs:

$$\text{AssA}(\alpha) = \frac{1}{|TP(\alpha)|} \sum_{(c,\,\hat{c})\,\in\, TP(\alpha)} \frac{|TPA(c,\hat{c})|}{|TPA(c,\hat{c})| + |FPA(c,\hat{c})| + |FNA(c,\hat{c})|}$$

where $TPA$, $FPA$, $FNA$ count frames where the pair is correctly associated, incorrectly associated, or missed respectively. **HOTA** at a single threshold is the geometric mean of the two, ensuring neither component can dominate:

$$\text{HOTA}(\alpha) = \sqrt{\text{DetA}(\alpha) \times \text{AssA}(\alpha)}$$

The final score is averaged uniformly across all thresholds, preventing a detector that is strong at loose IoU from masking poor localisation at strict thresholds:

$$\text{HOTA} = \frac{1}{|\mathcal{A}|} \sum_{\alpha \,\in\, \mathcal{A}} \text{HOTA}(\alpha)$$

**LocA** is the mean IoU of all matched pairs across thresholds, summarising spatial localisation quality independently of association.

In [ ]:
# Evaluate tracking performance of SORT and DeepSORT against ground truth annotations
tracking_results = {}
for tracker_name, outputs in [
    ("SORT",     sort_resolved_output),
    ("DeepSORT", deepsort_resolved_output),
]:
    for cam_id in CAMERAS:
        tracking_results[(tracker_name, cam_id)] = evaluate_tracking(
            ground_truth[cam_id], outputs[cam_id], frame_stride=FRAME_STRIDE,
        )

### Identity Metrics

In [ ]:
show_identity_table(tracking_results)

### HOTA Metrics

In [ ]:
show_hota_table(tracking_results)